# Stroke Prediction Notebook

A structured walkthrough for loading the dataset, preparing features, training multiclass models, and evaluating performance.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Data Loading and Setup

This study uses the publicly available [**Kaggle Healthcare Stroke Prediction Dataset**](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset). This dataset includes key medical histories, demographics, and lifestyle factors for patients, making it an ideal choice for building predictive models.

Load the dataset, remove unusable rows, and define the target for the modeling pipeline.

In [ ]:
np.random.seed(42)

df = pd.read_csv('healthcare-dataset-stroke-data.csv')
df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other']

print("Dataset loaded and cleaned.")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:\n{df.head()}")

True Target distribution:
 risk_tier
0    3426
1    1434
2     249
Name: count, dtype: int64
Train size (after Multiclass SMOTE): 7194 | Val size: 1022 | Test size: 511


### Feature Engineering: Multiclass Risk Tier Creation

Create a multiclass target variable based on stroke history and risk factors.

In [ ]:
# Engineering multiclass target before train-test split ensures stratification consistency
def define_risk_tier(row):
    if row['stroke'] == 1:
        return 2
    elif row['hypertension'] == 1 or row['heart_disease'] == 1 or row['age'] >= 60:
        return 1
    else:
        return 0

df['risk_tier'] = df.apply(define_risk_tier, axis=1)

X = df.drop(columns=['stroke', 'risk_tier'])
y = df['risk_tier']

print("Risk Tier distribution:\n", y.value_counts())
print(f"\nFeatures shape: {X.shape}")

### Preprocessing Pipeline Setup

Define numeric and categorical transformers to prepare features for modeling.

In [ ]:
numeric_features = ['age', 'avg_glucose_level', 'bmi']
categorical_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
passthrough_features = ['hypertension', 'heart_disease']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('pass', 'passthrough', passthrough_features)
    ])

### Data Splitting and Transformation

Split data into train/validation/test sets (70%/20%/10%) and apply preprocessing.

In [ ]:
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=1/3, random_state=42, stratify=y_temp
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_val_processed = preprocessor.transform(X_val_raw)
X_test_processed = preprocessor.transform(X_test_raw)

print(f"Train set: {X_train_processed.shape[0]} | Val set: {X_val_processed.shape[0]} | Test set: {X_test_processed.shape[0]}")

### Handling Class Imbalance with SMOTE

Apply Synthetic Minority Over-sampling Technique to balance the training set.

In [ ]:
# Apply SMOTE only to training set to prevent data leakage
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_processed, y_train)

X_val, y_val = X_val_processed, y_val.values
X_test, y_test = X_test_processed, y_test.values

print(f"Train size (after SMOTE): {X_train.shape[0]} | Val size: {X_val.shape[0]} | Test size: {X_test.shape[0]}")
print(f"\nBalanced training set distribution:\n{pd.Series(y_train).value_counts()}")

## 3. Model Training

Train two multiclass classifiers using tuned hyperparameters.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

mlp_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, activation='relu', solver='adam', random_state=42)
mlp_model.fit(X_train, y_train)

print("✓ Random Forest trained")
print("✓ Neural Network (MLP) trained")

## 4. Model Evaluation

Compare model performance using confusion matrices and classification reports.

In [ ]:
models = {'Random Forest': rf_model, 'Neural Network (MLP)': mlp_model}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_labels = ['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)']

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_labels, yticklabels=class_labels)
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted Risk Category')
    ax.set_ylabel('Actual Risk Category')
    
    print(f"\n{'='*40}\n{name} Performance\n{'='*40}")
    print(f"Validation Accuracy: {accuracy_score(y_val, model.predict(X_val)):.4f}")
    print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_labels))

plt.tight_layout()
plt.show()